# Shabaka Pulse — Industrial Facility Dataset

This notebook generates a synthetic dataset of 30 industrial facilities across Egypt participating in the Virtual Power Plant (VPP) demand-response program.

Facilities are grouped into three operational tiers based on their capacity to shed or absorb flexible load without disrupting core manufacturing outputs.

## 1. Setup

Importing core data processing libraries and setting a fixed random seed for reproducible dataset generation.

In [1]:
from pathlib import Path
from typing import Any, Dict, List
import numpy as np
import pandas as pd

# Set fixed seed to ensure deterministic results across execution runs
SEED = 42
np.random.seed(SEED)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

## 2. Define the three operational tiers

The VPP demand-response architecture categorizes participating facilities into three operational storage tiers:
- **Tier 1 (Hydraulic Storage)**: Desalination plants and water pumping stations. High ramp rates and fast response times; water storage reservoirs buffer operations so load shedding causes zero inventory or product degradation risk.
- **Tier 2 (Thermal Storage)**: Cold storage logistics and food preservation facilities. Moderate ramp rates; thermal inertia in refrigeration systems provides a buffer window before food safety bounds are breached.
- **Tier 3 (Batch Processing)**: Cement grinding mills and steel rolling mills. Slower ramp rates due to mechanical startup constraints, but large absolute flexible capacity (MW) enabled by buffer material silos between batch processing stages.

In [2]:
TIER_DEFINITIONS: List[Dict[str, Any]] = [
    {
        "tier": 1,
        "storage_type": "hydraulic",
        "industry_types": [
            "Desalination Plant",
            "Water Pumping Station",
        ],
        "max_flex_mw_range": (10.0, 35.0),
        "ramp_rate_mw_per_min_range": (1.50, 4.00),
    },
    {
        "tier": 2,
        "storage_type": "thermal",
        "industry_types": [
            "Cold Storage Logistics",
            "Food Preservation Facility",
        ],
        "max_flex_mw_range": (5.0, 20.0),
        "ramp_rate_mw_per_min_range": (0.50, 1.50),
    },
    {
        "tier": 3,
        "storage_type": "batch",
        "industry_types": [
            "Cement Grinding Mill",
            "Steel Rolling Mill",
        ],
        "max_flex_mw_range": (40.0, 100.0),
        "ramp_rate_mw_per_min_range": (0.10, 0.50),
    },
]

## 3. Define realistic Egyptian geographic anchors

Facilities are anchored near actual Egyptian industrial hubs and renewable corridors rather than scattered uniformly:
- **Aswan / Benban Corridor**: Southern Egypt industrial & water management facilities near solar infrastructure.
- **Suez / Ain Sokhna Industrial Zone**: Red Sea logistics, steel, and desalination hub.
- **Tenth of Ramadan / Nile Delta Hub**: Major industrial and food processing corridor.
- **Borg El Arab / Alexandria Zone**: Coastal industrial, cold chain, and chemical processing cluster.

In [3]:
GEOGRAPHIC_ANCHORS: List[Dict[str, Any]] = [
    {"name": "Aswan / Benban Zone", "lat": 24.43, "lon": 32.74},
    {"name": "Suez / Ain Sokhna Hub", "lat": 29.85, "lon": 32.55},
    {"name": "Tenth of Ramadan City", "lat": 30.30, "lon": 31.75},
    {"name": "Borg El Arab / Alexandria", "lat": 30.88, "lon": 29.58},
]

## 4. Generate the 30 facilities

Constructing a balanced dataset of 30 facilities across the three storage tiers. Each facility is assigned a unique identifier (`FAC-XXX`), name, industry type, storage classification, sampled flexible capacity (`max_flex_mw`), ramp rate (`ramp_rate_mw_per_min`), and geographical coordinates with small random jitter around an industrial anchor.

In [ ]:
def generate_facilities(
    num_facilities: int = 30,
    seed: int = SEED,
) -> pd.DataFrame:

    rng = np.random.default_rng(seed)
    records: List[Dict[str, Any]] = []

    # Track industry count per type to assign realistic sequential names
    industry_counters: Dict[str, int] = {}

    for i in range(num_facilities):
        # Cycle through tier definitions to ensure balanced representation across tiers
        tier_def = TIER_DEFINITIONS[i % len(TIER_DEFINITIONS)]
        
        industry_type = str(rng.choice(tier_def["industry_types"]))
        industry_counters[industry_type] = industry_counters.get(industry_type, 0) + 1
        seq_num = industry_counters[industry_type]

        facility_id = f"FAC-{i + 1:03d}"
        facility_name = f"{industry_type} #{seq_num}"

        # Sample capacity and ramp rates within tier-specific boundaries
        max_flex_mw = round(
            float(rng.uniform(*tier_def["max_flex_mw_range"])), 2
        )
        ramp_rate_mw_per_min = round(
            float(rng.uniform(*tier_def["ramp_rate_mw_per_min_range"])), 2
        )

        # Assign geographical anchor with small spatial Gaussian offset (~5 km)
        anchor = rng.choice(GEOGRAPHIC_ANCHORS)
        lat = round(float(anchor["lat"]) + float(rng.normal(loc=0.0, scale=0.05)), 4)
        lon = round(float(anchor["lon"]) + float(rng.normal(loc=0.0, scale=0.05)), 4)

        records.append(
            {
                "facility_id": facility_id,
                "facility_name": facility_name,
                "industry_type": industry_type,
                "max_flex_mw": max_flex_mw,
                "ramp_rate_mw_per_min": ramp_rate_mw_per_min,
                "storage_type": tier_def["storage_type"],
                "lat": lat,
                "lon": lon,
            }
        )

    df_facilities = pd.DataFrame(records)
    
    # Enforce exact column ordering
    column_order = [
        "facility_id",
        "facility_name",
        "industry_type",
        "max_flex_mw",
        "ramp_rate_mw_per_min",
        "storage_type",
        "lat",
        "lon",
    ]
    return df_facilities[column_order]


df_facilities = generate_facilities(num_facilities=30, seed=SEED)

## 5. Validate the generated dataset

Performing integrity and quality checks on the dataset:
- Verifying schema and inspecting top records (`.head(10)`).
- Counting facility distributions across industry and storage classifications.
- Inspecting summary statistics (`.describe()`) grouped by `storage_type` to confirm tier separation.
- Asserting unique facility IDs.

In [5]:
# Verify dataset shape and structure
display(df_facilities.head(10))

# Value counts for industry and storage types
display(df_facilities["industry_type"].value_counts().to_frame())
display(df_facilities["storage_type"].value_counts().to_frame())

# Grouped summary statistics confirming operational tier boundaries
grouped_stats = (
    df_facilities.groupby("storage_type")[["max_flex_mw", "ramp_rate_mw_per_min"]]
    .describe()
    .round(2)
)
display(grouped_stats)

# Assert no duplicate facility IDs exist
assert df_facilities["facility_id"].nunique() == len(
    df_facilities
), "Error: Duplicate facility IDs detected."

,facility_id,facility_name,industry_type,max_flex_mw,ramp_rate_mw_per_min,storage_type,lat,lon
0,FAC-001,Desalination Plant #1,Desalination Plant,20.97,3.65,hydraulic,30.9270,29.4824
1,FAC-002,Food Preservation Facility #1,Food Preservation Facility,16.42,1.29,thermal,30.8792,29.5373
2,FAC-003,Steel Rolling Mill #1,Steel Rolling Mill,95.61,0.36,batch,29.9064,32.5734
3,FAC-004,Desalination Plant #2,Desalination Plant,23.86,1.66,hydraulic,24.4739,32.7375
4,FAC-005,Cold Storage Logistics #1,Cold Storage Logistics,10.32,1.47,thermal,30.8723,29.5586
5,FAC-006,Steel Rolling Mill #2,Steel Rolling Mill,68.00,0.12,batch,24.4506,32.7615
6,FAC-007,Water Pumping Station #1,Water Pumping Station,34.19,2.31,hydraulic,30.2593,31.7808
7,FAC-008,Food Preservation Facility #2,Food Preservation Facility,6.95,0.98,thermal,24.3888,32.7725
8,FAC-009,Steel Rolling Mill #3,Steel Rolling Mill,89.96,0.38,batch,29.8772,32.5167
9,FAC-010,Desalination Plant #3,Desalination Plant,19.69,2.22,hydraulic,30.9236,29.5912


,count
industry_type,
Desalination Plant,7
Food Preservation Facility,6
Steel Rolling Mill,6
Cold Storage Logistics,4
Cement Grinding Mill,4
Water Pumping Station,3


,count
storage_type,
hydraulic,10
thermal,10
batch,10


max_flex_mw                                                   \
                   count   mean    std    min    25%    50%    75%    max   
storage_type                                                                
batch               10.0  79.18  13.19  59.89  67.65  82.66  88.85  95.61   
hydraulic           10.0  24.67   5.77  17.34  21.04  22.82  27.12  34.19   
thermal             10.0  10.41   3.25   5.11   8.52  10.38  12.04  16.42   

             ramp_rate_mw_per_min                                            
                            count  mean   std   min   25%   50%   75%   max  
storage_type                                                                 
batch                        10.0  0.32  0.09  0.12  0.29  0.33  0.38  0.44  
hydraulic                    10.0  2.51  0.63  1.66  2.19  2.46  2.86  3.65  
thermal                      10.0  1.06  0.34  0.59  0.80  1.14  1.31  1.47

## 6. Save the dataset

Writing the validated facility dataset to `data/facilities.csv`.

In [6]:
output_path = DATA_DIR / "facilities.csv"
df_facilities.to_csv(output_path, index=False)

summary_df = pd.DataFrame(
    [
        {
            "file_name": output_path.name,
            "path": str(output_path.resolve()),
            "total_rows": len(df_facilities),
            "total_columns": len(df_facilities.columns),
        }
    ]
)
display(summary_df)

,file_name,path,total_rows,total_columns
0,facilities.csv,D:\shabaka-pluse\data\facilities.csv,30,8
